# ML-KEM-768 KAT Notebook

Run NIST KAT vectors directly on KR260 hardware using the same driver logic as `ml_kem_kat_test.py`.

Recommended flow:
1. Run with `n_vectors = 1`.
2. Increase to `n_vectors = 100`.
3. Optionally run full KAT file.

In [ ]:
import os
import time

from ml_kem_driver import MLKem768, cycles_to_us


def parse_kat_file(path):
    vectors = []
    current = {}
    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if current:
                    vectors.append(current)
                    current = {}
                continue
            if "=" not in line:
                continue
            key, _, val = line.partition("=")
            key = key.strip()
            val = val.strip()
            try:
                current[key] = bytes.fromhex(val)
            except ValueError:
                current[key] = val
    if current:
        vectors.append(current)
    return vectors


In [ ]:
def run_kat(vectors, kem, stop_on_fail=False):
    n = len(vectors)
    pass_ct = 0
    fail_ct = 0
    first_fail = None
    totals = {"keygen": 0, "encaps": 0, "decaps": 0}
    fail_reasons = []

    t0 = time.monotonic()
    for i, v in enumerate(vectors):
        try:
            d = v["d"]
            z = v["z"]
            m = v["m"]
            pk_exp = v["pk"]
            sk_exp = v["sk"]
            ct_exp = v["ct"]
            ss_exp = v["ss"]
        except KeyError as e:
            fail_ct += 1
            fail_reasons.append((i, f"missing KAT field {e}"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        pk, sk, c_kg = kem.keygen(d, z)
        totals["keygen"] += c_kg
        if pk != pk_exp:
            fail_ct += 1
            fail_reasons.append((i, "pk mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue
        if sk != sk_exp:
            fail_ct += 1
            fail_reasons.append((i, "sk mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        ct, ss_enc, c_enc = kem.encaps(pk, m)
        totals["encaps"] += c_enc
        if ct != ct_exp:
            fail_ct += 1
            fail_reasons.append((i, "ct mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue
        if ss_enc != ss_exp:
            fail_ct += 1
            fail_reasons.append((i, "encaps ss mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        ss_dec, c_dec = kem.decaps(sk, ct)
        totals["decaps"] += c_dec
        if ss_dec != ss_exp:
            fail_ct += 1
            fail_reasons.append((i, "decaps ss mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        pass_ct += 1
        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[{i+1:4d}/{n}] pass (KG={c_kg}, Enc={c_enc}, Dec={c_dec})")

    wall = time.monotonic() - t0
    return {
        "n": n,
        "pass": pass_ct,
        "fail": fail_ct,
        "first_fail": first_fail,
        "reasons": fail_reasons,
        "totals": totals,
        "wall_seconds": wall,
    }


In [ ]:
script_dir = os.getcwd()
bitfile = os.environ.get("ML_KEM_BIT", os.path.join(script_dir, "ml_kem_bd.bit"))
kat_path = os.environ.get("KAT_FILE", os.path.join(script_dir, "KAT_768.txt"))
n_vectors = 1  # change to 100 or None for full file

print(f"bitfile: {bitfile}")
print(f"kat_path: {kat_path}")
print(f"n_vectors: {n_vectors}")


In [ ]:
vectors = parse_kat_file(kat_path)
if n_vectors is not None:
    vectors = vectors[:n_vectors]

with MLKem768(bitfile) as kem:
    result = run_kat(vectors, kem)

print("\n===== Summary =====")
print(f"PASS: {result['pass']} / {result['n']}")
print(f"FAIL: {result['fail']} / {result['n']}")

if result["fail"]:
    print(f"First failure index: {result['first_fail']}")
    for idx, reason in result["reasons"][:10]:
        print(f"  vec #{idx}: {reason}")

n_ok = result["pass"]
if n_ok > 0:
    t = result["totals"]
    avg_kg = t["keygen"] / n_ok
    avg_enc = t["encaps"] / n_ok
    avg_dec = t["decaps"] / n_ok
    avg_full = (t["keygen"] + t["encaps"] + t["decaps"]) / n_ok
    print("\nAverage hardware cycles (passed vectors):")
    print(f"  KeyGen: {avg_kg:.1f} cyc ({cycles_to_us(avg_kg):.1f} us)")
    print(f"  Encaps: {avg_enc:.1f} cyc ({cycles_to_us(avg_enc):.1f} us)")
    print(f"  Decaps: {avg_dec:.1f} cyc ({cycles_to_us(avg_dec):.1f} us)")
    print(f"  Full  : {avg_full:.1f} cyc ({cycles_to_us(avg_full):.1f} us)")

print(f"\nWall time: {result['wall_seconds']:.2f} s")
print(f"Per vector wall: {(result['wall_seconds'] / max(1, result['n'])) * 1000:.2f} ms")

assert result["fail"] == 0, "KAT regression failed"
